In [ ]:
# Qwen3.5-0.8B Stage A evaluation (eval-only notebook, no training paths)


In [ ]:
# Cell 0 — environment / P100 proof + mount check
import os, shutil, sys
from pathlib import Path

print('python', sys.version.split()[0], '(torch loads in Cell 1 with a P100-compatible build)')
!nvidia-smi --query-gpu=name,memory.total,memory.free,compute_cap --format=csv 2>/dev/null || nvidia-smi 2>/dev/null | head -20
print('working disk:', shutil.disk_usage('/kaggle/working'))
inp = sorted(str(p) for p in Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else []
print('/kaggle/input mounts:', inp if inp else 'NONE (real-PLE will abort until PLE dataset is attached)')
secret_value_0 = os.environ.get('HF_TOKEN')
secret_value_1 = os.environ.get('KAGGLE_API_TOKEN')
if not secret_value_0 or not secret_value_1:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        secret_value_0 = secret_value_0 or user_secrets.get_secret('HF_TOKEN')
        secret_value_1 = secret_value_1 or user_secrets.get_secret('KG_TOKEN')
    except Exception:
        pass
assert secret_value_0, 'set HF_TOKEN or attach the Kaggle HF_TOKEN secret'
assert secret_value_1, 'set KAGGLE_API_TOKEN or attach the Kaggle KG_TOKEN secret'
os.environ['HF_TOKEN'] = secret_value_0
os.environ['KAGGLE_API_TOKEN'] = secret_value_1


In [ ]:
# Cell 1 — deps (P100-compatible torch BEFORE first torch import)
import subprocess, sys
try:
    _cap=subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],capture_output=True,text=True,timeout=60).stdout.strip().splitlines()[0].strip()
except Exception: _cap=''
print('compute_cap:', _cap or 'unknown')
%pip install -q "transformers==5.17.0" datasets safetensors huggingface_hub matplotlib accelerate
if _cap.startswith('6.'):
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torch==2.5.1+cu118'],check=True)
    print('pinned P100 torch (cu118, sm_60 kernels)')
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torchvision==0.20.1+cu118'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torchaudio==2.5.1+cu118'],check=True)
    print('pinned matching torchvision+torchaudio (qwen3_5 modeling pulls both via media utils)')
import torch
assert torch.cuda.is_available(), 'need a GPU accelerator'
torch.zeros(1).cuda()  # fail fast if the build lacks sm_60 kernels
print('torch', torch.__version__, '| cap', torch.cuda.get_device_capability(0))
import transformers, datasets, safetensors, huggingface_hub
print(transformers.__version__, datasets.__version__, safetensors.__version__, huggingface_hub.__version__)

In [ ]:
# Cell 2 — single config block (edit here only)
from dataclasses import dataclass

@dataclass(frozen=True)
class Cfg:
    SOURCE_ID: str = 'Qwen/Qwen3.8-Flash-Next-FP8'
    TARGET_ID: str = 'Qwen/Qwen3.5-0.8B'
    MEM_DIM: int = 2560
    HIDDEN: int = 1024
    N_LAYERS: int = 24
    NGRAM: int = 3
    HEADS_PER_NGRAM: int = 8   # 16 address slots per token
    ROW_DIM: int = 160         # per-slot row dim; 16*160=2560
    ROWS_PER_PART: int = 2_500_012
    VOCAB_BASE: int = 20_000_000
    SEED: int = 1234           # PLE hash seed — never change
    EOS: int = 248044  # config <|endoftext|>: PLE-training terminator (NOT chat <|im_end|> 248046)
    VOCAB: int = 248320
    SEQ: int = 512
    VAL_FAST_TOKENS: int = 65536    # 128 x 512
    VAL_FULL_TOKENS: int = 524288   # 1024 x 512 (35B parity)
    DATASET_ID: str = 'HuggingFaceFW/fineweb-edu'  # FIRST experiment: FineWeb-Edu ONLY
    DATASET_CONFIG: str = 'sample-10BT'
    SMOKE_TOKENS: int = 5120    # correctness pass only (10 x 512); sweep stays OFF
    LR: float = 3e-5
    WD: float = 0.01
    WARMUP_FRAC: float = 0.05
    CKPTS: tuple = (100000, 250000, 500000)  # threshold-crossing (not exact multiples)
    PLACEMENTS: tuple = ((2,), (8,), (2, 8))  # zero-based IDX, 35B convention
    BRANCHES: int = 1
    GAMMA_INIT: float = 1e-3   # 0.0 = identity test mode


    EVAL_SEED: int = 1234
    EVAL_BS: int = 8
    EVAL_TARGET_S: int = 10800  # 3h soft target for Stage A benchmarks
    EVAL_HARD_S: int = 32400  # 9h hard guard with persist margin (well below 12h limit)
    EVAL_MMLU_N: int = 1000
    EVAL_SQA_N: int = 1000
    ARMS_500K: tuple = ('disabled', 'random', 'permuted', 'real')  # controlled comparison
    ARM_1M: str = 'real-1m'  # exploratory only, never in the causal set

C = Cfg()
def describe(layers): return f"IDX {list(layers)} = HUMAN {[l+1 for l in layers]}"
print(C)
print('placements:', ' / '.join(describe(l) for l in C.PLACEMENTS))
for sites, R in [(1,1),(2,1),(1,4),(2,4)]:
    n = sites*(R*C.MEM_DIM*C.HIDDEN + C.MEM_DIM*C.HIDDEN) + sites*R + sites
    print(f'sites={sites} R={R}: ~{n/1e6:.2f}M trainable')
print('controlled arms @500K:', C.ARMS_500K, '| exploratory:', C.ARM_1M)


In [ ]:
# Cell 3 — EXACT source addressing (port of src/qwen36_ple/hashing.py — do not modify)
import math
import torch

MASK64=(1<<64)-1; GAMMA=0x9E3779B97F4A7C15; M1=0xBF58476D1CE4E5B9; M2=0x94D049BB133111EB; LPRIME=10007

def splitmix64(v):
    v=(v+GAMMA)&MASK64; v=((v^(v>>30))*M1)&MASK64; v=((v^(v>>27))*M2)&MASK64; return (v^(v>>31))&MASK64

def layer_multipliers(vocab, ngram, ple_idx=0, seed=1234):
    mmax=((1<<63)-1)//max(vocab,1); hb=max(1,mmax//2); base=seed+LPRIME*ple_idx
    return tuple(2*(splitmix64((base+GAMMA*(i+1))&MASK64)%hb)+1 for i in range(ngram))

def is_prime(v):
    if v<2: return False
    if v%2==0: return v==2
    return all(v%d for d in range(3, math.isqrt(v)+1, 2))

def nth_prime_after(s, k):
    p=s
    for _ in range(k):
        p+=1
        while not is_prime(p): p+=1
    return p

def head_layout(ngram=3, hpn=8, base=20_000_000, ple_idx=0):
    n=(ngram-1)*hpn
    sizes=tuple(nth_prime_after(base-1, ple_idx*n+h+1) for h in range(n))
    off=[]; t=0
    for s in sizes: off.append(t); t+=s
    return sizes, tuple(off)

def shift_right_ignore_eos(ids, shift, eos):
    if shift==0: return ids
    B,L=ids.shape; pos=torch.arange(L, device=ids.device)
    eos_pos=torch.where(ids==eos, pos, -1); prev_inc=torch.cummax(eos_pos,1).values
    prev=torch.cat([eos_pos.new_full((B,1),-1), prev_inc[:,:-1]],1)
    inseg=pos.unsqueeze(0)-(prev+1); src=pos-shift
    sh=ids.gather(1, src.clamp_min(0).unsqueeze(0).expand(B,-1))
    valid=(inseg>=shift)&(src.unsqueeze(0)>=0)
    return torch.where(valid, sh, ids.new_full((), eos))

def ngram_indices(ids, eos_token_id=248044, vocab_size=248320, ngram_size=3, heads_per_ngram=8,
                    vocab_size_base=20_000_000, ple_layer_index=0, seed=1234):
    ids=ids.long()
    mult=torch.tensor(layer_multipliers(vocab_size, ngram_size, ple_layer_index, seed), device=ids.device)
    sizes, offs=head_layout(ngram_size, heads_per_ngram, vocab_size_base, ple_layer_index)
    sizes=torch.tensor(sizes, device=ids.device); offs=torch.tensor(offs, device=ids.device)
    sh=[shift_right_ignore_eos(ids,s,eos_token_id) for s in range(ngram_size)]
    blocks=[]
    for ng in range(2, ngram_size+1):
        st=(ng-2)*heads_per_ngram; mixed=sh[0]*mult[0]
        for p in range(1,ng): mixed=torch.bitwise_xor(mixed, sh[p]*mult[p])
        blocks.append(torch.remainder(mixed.unsqueeze(-1), sizes[st:st+heads_per_ngram])+offs[st:st+heads_per_ngram])
    return torch.cat(blocks,-1)  # [B,L,16] GLOBAL PLE addresses, never raw token ids

_a=ngram_indices(torch.tensor([[1,2,3,4,5]])); _b=ngram_indices(torch.tensor([[1,2,3,4,5]]))
assert torch.equal(_a,_b) and _a.shape==(1,5,16)
SIZES, OFFS = head_layout()
print('hash ok; slots/head addrs e.g.', tuple(_a[0,2,:4].tolist()), '| head0 range', (OFFS[0], OFFS[0]+SIZES[0]))
def addresses(token_cpu):
    '''ONLY path from tokens to PLE rows: exact source hashes -> global head addresses.'''
    return ngram_indices(token_cpu, eos_token_id=C.EOS, vocab_size=C.VOCAB,
                         ngram_size=C.NGRAM, heads_per_ngram=C.HEADS_PER_NGRAM,
                         vocab_size_base=C.VOCAB_BASE, ple_layer_index=0, seed=C.SEED)

In [ ]:
# Cell 4 — shared-value reader (hidden=1024; reductions stay FP32 so fp16 backbone is safe)
import math
import torch
from torch import nn

def rms_norm(x, eps=1e-6):  # always FP32 reduction, cast back: fp16-safe
    return x.float().mul(torch.rsqrt(x.float().square().mean(-1, keepdim=True)+eps)).to(x.dtype)

class SharedValueReader(nn.Module):
    def __init__(self, mem_dim=2560, hidden=1024, branches=1, gamma=0.0):
        super().__init__(); self.mem_dim=mem_dim; self.hidden=hidden; self.branches=branches
        self.keys=nn.ModuleList(nn.Linear(mem_dim, hidden, bias=False) for _ in range(branches))
        self.value=nn.Linear(mem_dim, hidden, bias=False)
        self.beta=nn.Parameter(torch.zeros(branches))
        self.gamma=nn.Parameter(torch.tensor(float(gamma)))
        self.last_gate=None
    def stats(self):
        if self.last_gate is None: return None
        g=self.last_gate.float()
        return {'mean':g.mean().item(),'std':g.std(correction=0).item(),'near_zero':(g<0.01).float().mean().item()}
    def forward(self, h, m):
        assert h.shape[:-1]==m.shape[:-1], (h.shape, m.shape)
        h_dtype=h.dtype; h=h.float(); m=m.float()  # backbone may be fp16; reader computes FP32
        q=rms_norm(h); v=self.value(m); gs=[]
        for b,proj in enumerate(self.keys):
            k=rms_norm(proj(m))
            s=(q.float()*k.float()).sum(-1)/math.sqrt(self.hidden)
            gs.append(torch.sigmoid(s+self.beta[b].float()).to(v.dtype))
        g=torch.stack(gs,0)
        o=(g.unsqueeze(-1)*v.unsqueeze(0)).mean(0)
        self.last_gate=g.detach()
        return (h+self.gamma*o).to(h_dtype)  # residual back to backbone dtype; params stay FP32

_r=SharedValueReader(gamma=0.0); _h=torch.randn(1,4,1024); _m=torch.randn(1,4,2560)
assert torch.equal(_r(_h,_m),_h)
print('reader identity ok; R=1 params:', sum(p.numel() for p in _r.parameters()))

In [ ]:
# Cell 5 — injection hooks (IDX convention) + layer helper
import torch
from torch import nn

def decoder_layers(model):
    for path in ['model.layers','language_model.layers','transformer.h']:
        o=model
        try:
            for a in path.split('.'): o=getattr(o,a)
            if len(o)==24 or len(o)>0: return o
        except Exception: pass
    raise RuntimeError('decoder layers not found')

class ReaderInjection(nn.Module):
    '''layers = zero-based IDX list, exactly like the 35B run (e.g. (2,) = third block).'''
    def __init__(self, model, layers, mem_dim=2560, hidden=1024, branches=1, gamma=0.0):
        super().__init__()
        self.idx=tuple(layers)
        self.readers=nn.ModuleDict({str(l):SharedValueReader(mem_dim,hidden,branches,gamma) for l in layers})
        self.memory=None; self.handles=[]
        dec=decoder_layers(model)
        assert len(dec)==C.N_LAYERS, f'decoder count {len(dec)} != {C.N_LAYERS} — wrong hook target'
        for l in layers:
            self.handles.append(dec[l].register_forward_pre_hook(self._hook(str(l)), with_kwargs=True))
        print(f'inject at IDX {list(layers)} = HUMAN {[l+1 for l in layers]}')
    def _hook(self,name):
        def fn(mod,args,kw):
            if self.memory is None: return args,kw
            m=self.memory
            L=args[0].shape[1] if args else kw['hidden_states'].shape[1]
            if m.shape[1]!=L: m=m[:,:L]
            if args: return (self.readers[name](args[0],m),*args[1:]),kw
            kw['hidden_states']=self.readers[name](kw['hidden_states'],m); return args,kw
        return fn
    def set_memory(self,m): self.memory=m
    def close(self):
        [h.remove() for h in self.handles]; self.handles.clear()

print('injection ok')

In [ ]:
# Cell 6 — PLE stores: real (mount-only, exact scale or abort) + calibrated controls
import json, os
from pathlib import Path
import torch
from safetensors import safe_open

PLE_TMPL='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.shard_{p}.weight'
PLE_SCALE='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.weight_scale'

def rss_mb():
    '''Host RSS in MiB (Linux /proc; -1 if unavailable). Proves bounded RAM.'''
    try:
        with open('/proc/self/status') as _f:
            for _line in _f:
                if _line.startswith('VmRSS:'): return float(_line.split()[1])/1024
    except Exception: pass
    try:
        import resource; return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024
    except Exception: return -1.0

def find_ple_manifests():
    hits=sorted(Path('/kaggle/input').glob('*/manifest.json'))+sorted(Path('/kaggle/input').glob('*/*/manifest.json'))
    out=[]
    for h in hits:
        try:
            m=json.loads(h.read_text())
            if isinstance(m, dict) and 'parts' in m: out.append(h)
        except Exception: pass
    return out

class MountPLE:
    '''Real frozen PLE (row-level mmap, 35B-validated). REQUIRES /kaggle/input mount. Never downloads. Never caches parts: each lookup fetches ONLY needed rows via safetensors get_slice runs, so host RAM stays bounded after all 128 parts are touched.'''
    def __init__(self, manifest=None):
        manifests=[Path(manifest)] if manifest else find_ple_manifests()
        if not manifests or not all(p.exists() for p in manifests):
            raise RuntimeError('Real-PLE ABORT: no /kaggle/input PLE dataset attached. Attach the pinned shards+manifest.json dataset(s) first; refusing to download 48.7 GiB into /kaggle/working.')
        self.part_paths={}; revs=set()
        for mp in manifests:
            m=json.loads(mp.read_text())
            if not revs: self.rpp=m.get('rows_per_part',2500012); self.rd=m.get('row_dim',160)
            revs.add(m.get('ple_revision','unknown'))
            for k,v in m['parts'].items():
                p=mp.parent/v
                if not p.exists(): continue  # each dataset holds only its own shards
                if int(k) in self.part_paths:
                    assert self.part_paths[int(k)].name==p.name, f'part {k} filename clash'
                    continue
                self.part_paths[int(k)]=p
        assert len(revs)==1, f'mixed PLE revisions: {revs}'
        self.ple_revision=revs.pop()
        assert len(self.part_paths)==128, f'need all 128 parts, have {len(self.part_paths)}'
        missing=[str(p) for p in self.part_paths.values() if not p.exists()]
        if missing: raise RuntimeError(f"Real-PLE ABORT: {len(missing)} shard files missing, e.g. {missing[0]}")
        self.scale=self._resolve_scale()  # exact scale or abort — no fallback constant
        self.calls=0; self.rows_read=0
        self.parts_touched=set()  # cumulative DISTINCT parts; no part tensors ever held
        print(f'mounted PLE rev={self.ple_revision} parts={len(self.part_paths)} scale={self.scale}')
    def _resolve_scale(self):
        for f in sorted(set(self.part_paths.values())):
            try:
                with safe_open(f, framework='pt', device='cpu') as fh:
                    if PLE_SCALE in fh.keys():
                        return float(fh.get_tensor(PLE_SCALE).float().mean())
            except Exception: pass
        raise RuntimeError('Real-PLE ABORT: weight_scale tensor not found in pinned shards/index. Refusing hardcoded fallback.')
    def stats(self):
        return {'calls': self.calls, 'rows_read': self.rows_read,
                'parts_touched': len(self.parts_touched), 'held_part_tensors': 0,
                'scale': self.scale, 'rss_MiB': round(rss_mb(), 1)}
    @staticmethod
    def tensor_name(part): return PLE_TMPL.format(p=part)
    def lookup(self, indices):  # indices = GLOBAL head addresses [..,16] from ngram_indices()
        '''Row-level mmap reads: group deduped addresses by part (sorted), fetch ONLY
        needed rows as contiguous get_slice runs, dequantize the gathered rows. Full
        part tensors (~381 MiB each) are never materialized or cached.'''
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)  # dedup repeated addresses
        parts=torch.div(uniq, self.rpp, rounding_mode='floor'); local=uniq%self.rpp
        table=torch.empty(uniq.numel(), self.rd, dtype=torch.float32)
        for p in torch.unique(parts).tolist():  # ascending part order (page-friendly)
            pos=torch.nonzero(parts==p).flatten()
            lrows=local.index_select(0, pos); srows, sidx=torch.sort(lrows)
            runs=[]; a=int(srows[0]); prev=a
            for r in srows[1:].tolist():
                if r==prev+1: prev=r
                else: runs.append((a, prev+1)); a=prev=r
            runs.append((a, prev+1))
            path=self.part_paths.get(p)
            if path is None: raise FileNotFoundError(f'PLE part {p} not in manifest')
            with safe_open(str(path), framework='pt', device='cpu') as fh:
                sl=fh.get_slice(self.tensor_name(p))
                got=torch.cat([sl[x:y].to(torch.float32) for x, y in runs])*self.scale
            table.index_copy_(0, pos.index_select(0, sidx), got)  # got aligns with srows
        self.calls+=1; self.rows_read+=uniq.numel(); self.parts_touched.update(torch.unique(parts).tolist())
        return table[inv].reshape(*shape, self.rd).flatten(-2)  # [..,16,160]->[..,2560]

def permute_addresses(addrs, seed=777):
    '''Deterministic per-head bijective address permutation (shared by builder + store).'''
    sizes, offs=head_layout()
    S=torch.tensor(sizes); O=torch.tensor(offs)
    A=[]; B=[]
    for h,(s,o) in enumerate(zip(sizes, offs)):
        a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
        b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
        assert a%s!=0, 'A must be coprime to prime head size'
        A.append(a); B.append(b)
    A=torch.tensor(A); B=torch.tensor(B)
    f=addrs.long().cpu()
    H=f.shape[-1]
    O=O[:H]; A=A[:H]; B=B[:H]; S=S[:H]
    return O+((f-O)*A+B)%S

class RandomPLE:
    '''Calibrated per-head deterministic control: same global address -> same 160-d row, every call.
    Means/stds are per-head scalars measured from real rows in the frozen working set.
    16 rows concatenate to 2560-d. No 0.06 fallback.'''
    def __init__(self, head_means, head_stds, seed=0, row_dim=160):
        assert len(head_means)==16 and len(head_stds)==16, 'need 16 per-head stats'
        self.means=[float(m) for m in head_means]
        self.stds=[float(s) for s in head_stds]
        assert all(s>0 for s in self.stds), 'stds must be positive (calibrated)'
        self.seed=seed; self.rd=row_dim
        _sizes,_offs=head_layout()
        self._S=list(_sizes); self._O=list(_offs)
    def _head_of(self, a):
        for h in range(16):
            if self._O[h]<=a<self._O[h]+self._S[h]: return h
        raise ValueError('address outside head ranges')
    def _rows_for(self, uniq, heads):
        rows=[]
        for a,h in zip(uniq.tolist(), heads.tolist()):
            g=torch.Generator(); g.manual_seed((self.seed*1000003+int(a))%2**63)
            rows.append(torch.randn(self.rd, generator=g)*self.stds[h]+self.means[h])
        return torch.stack(rows)
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)
        heads=torch.tensor([self._head_of(int(a)) for a in uniq.tolist()])
        table=self._rows_for(uniq, heads)
        return table[inv].reshape(*shape, self.rd).flatten(-2)

def calibrate_head_stats(compact, n_per_head=4096, seed=0):
    '''Measure per-head mean/std from representative real rows in the compact working set.
    Samples n_per_head rows per head from the frozen prefix (train + full-val).'''
    sizes, offs=head_layout()
    addrs=compact.addrs.to(torch.int64)
    g=torch.Generator().manual_seed(seed)
    means=[]; stds=[]
    for h in range(16):
        lo=offs[h]; hi=offs[h]+sizes[h]
        pos=torch.nonzero((addrs>=lo)&(addrs<hi)).flatten()
        assert len(pos)>=n_per_head, f'head {h} only {len(pos)} rows'
        pick=pos[torch.randint(0,len(pos),(n_per_head,),generator=g)]
        rows=compact.rows.index_select(0, pick).to(torch.float32)*compact.scale
        means.append(float(rows.mean()))
        stds.append(float(rows.std(correction=0)))
    print('calibrated head means:', [round(m,6) for m in means])
    print('calibrated head stds:', [round(s,6) for s in stds])
    return means, stds

class PermutedPLE:
    '''Bijective per-head permutation: off_h + ((a-off_h)*A_h + B_h) % size_h.
    Preserves head ranges (sizes are prime, A_h % size_h != 0 so gcd=1 i.e. coprime) and table distribution.
    Wraps a compact working-set cache, never /kaggle/input random reads during training.'''
    def __init__(self, base, seed=777):
        self.b=base; self.seed=seed
        sizes, offs=head_layout()
        self.S=torch.tensor(sizes); self.O=torch.tensor(offs)
        A=[]; B=[]
        for h,(s,o) in enumerate(zip(sizes, offs)):
            a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
            b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
            assert a%s!=0, 'A must be coprime to prime head size'
            A.append(a); B.append(b)
        self.A=torch.tensor(A); self.B=torch.tensor(B)
    def lookup(self, indices):
        H=indices.shape[-1]; f=indices.long().cpu()
        O=self.O[:H]; A=self.A[:H]; B=self.B[:H]; S=self.S[:H]
        return self.b.lookup((O+((f-O)*A+B)%S).to(indices.device) if indices.is_cuda else (O+((f-O)*A+B)%S))

class CompactPLE:
    '''Compact working set for one frozen token prefix: sorted int32 addresses + fp8 rows (mmap).
    Bit-exact vs MountPLE: same bytes, same scale, same dequant formula. No 48.7 GiB traffic.'''
    def __init__(self, directory):
        import json as _json
        d=Path(directory)
        meta=_json.loads((d/'compact.json').read_text())
        n=meta['address_count']
        self.addrs=torch.from_file(str(d/'addrs.u32'), shared=True, size=n, dtype=torch.int32)
        raw=torch.from_file(str(d/'rows.u8'), shared=True, size=n*meta['row_dim'], dtype=torch.uint8)
        self.rows=raw.view(torch.float8_e4m3fn).view(n, meta['row_dim'])
        self.scale=float(meta['scale']); self.meta=meta
        self.ple_revision=meta.get('ple_revision')
        print('compact PLE: %d rows, %.2f GiB mapped, scale=%g' % (n, (d/'rows.u8').stat().st_size/1024**3, self.scale))
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        assert bool((flat>=0).all()) and int(flat.max())<2**31
        f32=flat.to(torch.int32)
        pos=torch.searchsorted(self.addrs, f32)
        posc=pos.clamp(max=len(self.addrs)-1)
        assert bool((self.addrs[posc]==f32).all()), 'compact miss: address outside frozen prefix'
        out=self.rows.index_select(0, posc).to(torch.float32)*self.scale
        return out.reshape(*shape, self.rows.shape[-1]).flatten(-2)



print('stores ok; manifests:', [str(p) for p in find_ple_manifests()])


In [ ]:
# Cell 7 — tokenizer verification: SEMANTIC id-space check (SPEC #15)
# Pinned-rev forensics: both model.vocab = 248044 entries with 0 id diffs; source-only
# ids are 7 audio added-tokens (248070-248076) above the target max: no collision.
# Config eos = 248044 (<|endoftext|>) on BOTH. AutoTokenizer.eos_token_id may report
# chat <|im_end|> 248046 instead: a chat-template default, NOT the PLE-training
# terminator. Hashing keeps training constants (vocab 248320 / eos 248044 / seed 1234).
import json
from transformers import AutoTokenizer
from huggingface_hub import HfApi, hf_hub_download

tok=secret_value_0; assert tok, 'Attach HF_TOKEN in Settings -> Secrets'
api=HfApi(token=tok)
trev=api.model_info(C.TARGET_ID).sha; srev=api.model_info(C.SOURCE_ID).sha
print('target rev', trev[:12], '| source rev', srev[:12])
tt=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev)
st=AutoTokenizer.from_pretrained(C.SOURCE_ID, token=tok, revision=srev)
def raw_vocab(path):
    tj=json.load(open(path, encoding='utf-8'))
    m=dict(tj['model']['vocab'])
    for a in tj.get('added_tokens', []): m[a['content']]=a['id']
    return m
tm=raw_vocab(hf_hub_download(C.TARGET_ID,'tokenizer.json',revision=trev,token=tok))
sm=raw_vocab(hf_hub_download(C.SOURCE_ID,'tokenizer.json',revision=srev,token=tok))
diff={k for k in tm if k in sm and tm[k]!=sm[k]}
extra_t={k for k in tm if k not in sm}
smax=max(tm.values())
src_only={k: sm[k] for k in sm if k not in tm}
print('mapping diffs:', len(diff), '| target-only:', len(extra_t), '| source-only:', len(src_only))
assert not diff and not extra_t, 'target id space must match source ids exactly (native addressing)'
assert all(v>smax for v in src_only.values()), 'source-only ids must sit above target range'
assert sm.get('<|endoftext|>')==C.EOS and tm.get('<|endoftext|>')==C.EOS, 'training eos must be <|endoftext|> both sides'
probe='The quick brown fox 0123456789 function print(){} <|im_start|>x<|im_end|>'
assert tt.encode(probe, add_special_tokens=False)==st.encode(probe, add_special_tokens=False), 'probe encodings differ — STOP'
print('NATIVE token-ID addressing VALID: identical ids; training terminator eos =', C.EOS)


In [ ]:
# Cell 8 — immutable validation artifacts (built ONCE, never inside training) + FineWeb-Edu-only stream
import hashlib, json
from array import array
from pathlib import Path
from datasets import load_dataset

WORK=Path('/kaggle/working/ple-08b'); WORK.mkdir(parents=True, exist_ok=True)
VALDIR=WORK/'val-frozen-v1'; VALDIR.mkdir(exist_ok=True)

def _write_val(name, tokens_u32, meta_extra):
    tp=VALDIR/f'tokens-{name}.uint32le'; mp=VALDIR/f'validation-{name}.json'
    raw=tokens_u32.tobytes(); digest=hashlib.sha256(raw).hexdigest()
    meta={'name':name,'token_count':len(tokens_u32),'seq':512,'count':len(tokens_u32)//512,
            'tokens_sha256':digest, **meta_extra}
    if mp.exists():
        old=json.loads(mp.read_text())
        if old!=meta or tp.read_bytes()!=raw:
            raise RuntimeError(f'Immutable validation {name} differs — refusing to overwrite')
        print(f"reuse frozen val-{name} sha={digest[:16]} n={len(tokens_u32)}"); return meta
    tp.write_bytes(raw); mp.write_text(json.dumps(meta,indent=2)); print(f'wrote frozen val-{name} sha={digest[:16]}')
    return meta

def build_validation_artifacts(tok):
    '''Prefix-consistent: stream 524288 FineWeb-Edu tokens once; fast = first 65536 slice.'''
    from huggingface_hub import HfApi
    import os
    api=HfApi(token=secret_value_0)
    drev=api.dataset_info(C.DATASET_ID).sha; trev2=api.model_info(C.TARGET_ID).sha
    need_full=VALDIR/'validation-full.json'; need_fast=VALDIR/'validation-fast.json'
    if need_full.exists() and need_fast.exists():
        return json.loads(need_fast.read_text()), json.loads(need_full.read_text())
    ds=load_dataset(C.DATASET_ID, C.DATASET_CONFIG, split='train', streaming=True, revision=drev)
    arr=array('I'); docs=0
    for row in ds:
        docs+=1; t=row.get('text') or ''
        if t.strip(): arr.extend(tok.encode(t, add_special_tokens=False)); arr.append(C.EOS)  # training terminator
        if len(arr)>=C.VAL_FULL_TOKENS: del arr[C.VAL_FULL_TOKENS:]; break
    assert len(arr)==C.VAL_FULL_TOKENS, len(arr)
    base={'dataset':C.DATASET_ID,'config':C.DATASET_CONFIG,'dataset_rev':drev,'target_rev':trev2,'docs':docs,'skip_docs':docs}
    fast_arr=array('I', arr[:C.VAL_FAST_TOKENS])
    mf=_write_val('fast', fast_arr, base); mF=_write_val('full', arr, base)
    return mf, mF

def load_validation(name):
    m=json.loads((VALDIR/f'validation-{name}.json').read_text())
    raw=(VALDIR/f'tokens-{name}.uint32le').read_bytes()
    assert hashlib.sha256(raw).hexdigest()==m['tokens_sha256'], 'val checksum mismatch'
    a=array('I'); a.frombytes(raw)
    import torch
    return torch.tensor(a,dtype=torch.long).view(-1,512), m



print('build with build_validation_artifacts(tok); read with load_validation("fast"/"full")')

In [ ]:
# Cell 9 — frozen Qwen3.5-0.8B via full-config VLM-compat CausalLM (vision frozen, text path): float16 FIRST
import os, torch
from transformers import AutoConfig, AutoModelForCausalLM

tok=secret_value_0; assert tok, 'Attach HF_TOKEN in Settings -> Secrets'
from huggingface_hub import HfApi
trev=HfApi(token=tok).model_info(C.TARGET_ID).sha
cfg=AutoConfig.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)
assert getattr(cfg,'model_type',None)=='qwen3_5', getattr(cfg,'model_type',None)
tconf=cfg.text_config  # dims ONLY — never pass as config= (strips auto_map, breaks class resolution)
print('hidden',tconf.hidden_size,'layers',tconf.num_hidden_layers,'vocab',tconf.vocab_size,'arch',type(cfg).__name__)
assert tconf.hidden_size==1024 and tconf.num_hidden_layers==24
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)

try:
    from transformers.models.qwen3_5 import Qwen3_5ForCausalLM as _Q
    print('direct qwen3_5 import ok:', _Q.__name__)
except Exception:
    import traceback; traceback.print_exc()
    raise RuntimeError('qwen3_5 modeling import failed — true cause above')
ACTIVE_DTYPE=None; model=None
for dt in [torch.float16, torch.float32]:
    try:
        m=AutoModelForCausalLM.from_pretrained(C.TARGET_ID, revision=trev, token=tok,
            trust_remote_code=True, device_map={'':0}, torch_dtype=dt, low_cpu_mem_usage=True)
        m.eval(); m.requires_grad_(False)
        assert sum(1 for p in m.parameters() if p.requires_grad)==0
        ids=tokenizer('The quick brown fox jumps over the lazy dog. '*8, return_tensors='pt').input_ids[:,:64].cuda()
        with torch.inference_mode():
            lg=m(input_ids=ids,use_cache=False).logits
        assert torch.isfinite(lg.float()).all(), 'non-finite logits'
        model=m; ACTIVE_DTYPE=dt; print(f'frozen load OK in {dt} footprint={m.get_memory_footprint()/1024**3:.2f} GiB')
        print('model class:', type(m).__name__, '| decoder blocks:', len(decoder_layers(m)))
        del ids, lg; break
    except Exception as e:
        print(f'{dt} rejected: {str(e)[:200]}')
        try: del m
        except Exception: pass
assert model is not None and ACTIVE_DTYPE is not None
print('backbone = ACTIVE_DTYPE:', ACTIVE_DTYPE)
print('reader params/compute = FP32 (AdamW FP32; norms/scores FP32)')
print('PLE dequant/output = FP32')
print('reader residual output is cast back to backbone dtype')

In [ ]:
# Cell 11b — validation prep: build immutable artifacts ONCE, then load both (idempotent)
mf, mfull = build_validation_artifacts(tokenizer)
val_fast, _ = load_validation("fast")
val_full, _ = load_validation("full")

print("validation ready")
print("fast:", mf["tokens_sha256"])
print("full:", mfull["tokens_sha256"])


In [ ]:
# NOTE: live hook-safe copies of _gen/_logprob_cont run in c-stageA; these defs are dormant spec.
PRED = []
import json, re, subprocess, sys, time, zlib
import torch, torch.nn.functional as F
EVAL_SEED = 1234
EVAL_BS = 8
def _ds_rev(ds_id):
    from huggingface_hub import HfApi
    import os
    try:
        return HfApi(token=secret_value_0).dataset_info(ds_id).sha
    except Exception:
        return None
def _load_split(ds_id, split, seed=EVAL_SEED):
    from datasets import load_dataset
    rev = _ds_rev(ds_id)
    kw = {'revision': rev} if rev else {}
    ds = load_dataset(ds_id, split=split, **kw)
    return ds, rev
def _subset(rows, n, seed=EVAL_SEED):
    import random
    rows = list(rows)
    if len(rows) <= n:
        return rows
    rng = random.Random(seed)
    idx = list(range(len(rows)))
    rng.shuffle(idx)
    return [rows[i] for i in sorted(idx[:n])]
def _prep_tok():
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = C.EOS
def _gen(prompts, max_new, arms):
    # greedy completions per arm
    import torch
    _prep_tok()
    out = {a['name']: [] for a in arms}
    for i in range(0, len(prompts), EVAL_BS):
        chunk = prompts[i:i + EVAL_BS]
        enc = tokenizer(chunk, return_tensors='pt', padding=True, add_special_tokens=False)
        ids = enc['input_ids']
        mask = enc['attention_mask']
        for a in arms:
            a['inj'].set_memory(a['store'].lookup(addresses(ids)).to('cuda') if a['store'] is not None else None)
            with torch.inference_mode():
                g = model.generate(input_ids=ids.to('cuda'), attention_mask=mask.to('cuda'),
                                   max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.pad_token_id,
                                   use_cache=True)
            txts = tokenizer.batch_decode(g[:, ids.shape[1]:], skip_special_tokens=True)
            out[a['name']].extend(txts)
    return out
def _logprob_cont(prompt, conts, arm):
    # summed logprob of each continuation after prompt
    import torch
    _prep_tok()
    p = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)['input_ids']
    outs = []
    with torch.inference_mode():
        for c in conts:
            t = tokenizer(c, return_tensors='pt', add_special_tokens=False)['input_ids']
            ids = torch.cat([p, t], dim=1)
            arm['inj'].set_memory(arm['store'].lookup(addresses(ids)).to('cuda') if arm['store'] is not None else None)
            lg = model(input_ids=ids.to('cuda'), use_cache=False).logits.float()
            lp = torch.log_softmax(lg[0, p.shape[1] - 1:-1], dim=-1)
            outs.append(lp.gather(1, t[0].to('cuda').unsqueeze(1)).sum().item())
    return outs
def _exec_check(code, timeout_s=10):
    try:
        r = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, timeout=timeout_s)
        return r.returncode == 0, (r.stderr or '')[-300:]
    except subprocess.TimeoutExpired:
        return False, 'timeout'
def _norm(s):
    s = s.lower().strip()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return re.sub(r'[^a-z0-9]', '', s)
def t_humaneval(arms, budget_ts):
    try:
        try:
            from evalplus.data import get_human_eval_plus
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'evalplus'], timeout=600, check=False)
            from evalplus.data import get_human_eval_plus
        probs = get_human_eval_plus()
        ds_id, ds_rev, plus = 'evalplus/HumanEvalPlus', 'builtin', True
    except Exception as e:
        ds, rev = _load_split('openai_humaneval', 'test')
        probs = {r['task_id']: {'prompt': r['prompt'], 'test': r['test'], 'entry_point': r['entry_point']} for r in ds}
        ds_id, ds_rev, plus = 'openai_humaneval', rev, False
    items = sorted(probs.items())
    gens = _gen([p['prompt'] for _, p in items], 512, arms)
    res, det = {}, {'dataset': ds_id, 'rev': str(ds_rev), 'plus': plus, 'n': len(items)}
    for a in arms:
        ok = sk = 0
        for (tid, p), comp in zip(items, gens[a['name']]):
            if time.perf_counter() > budget_ts:
                break
            code = p['prompt'] + '\n' + comp + '\n' + p['test'] + '\ncheck(' + p['entry_point'] + ')'
            try:
                passed, _ = _exec_check(code)
                ok += int(passed)
            except Exception:
                sk += 1
        res[a['name']] = {'pass': ok, 'n': len(items), 'acc': ok / max(1, len(items)), 'exec_errors': sk}
    return 'humaneval+', res, det
def _load_first(cands, split):
    errs = []
    for _cid in cands:
        try:
            _ds, _rev = _load_split(_cid, split)
            print('dataset resolved: ' + _cid, flush=True)
            return _ds, _rev, _cid
        except Exception as e:
            errs.append(_cid + ': ' + str(e)[:120])
    raise RuntimeError('no candidate resolved: ' + ' | '.join(errs))
def t_simpleqa(arms, budget_ts, n=1000):
    ds, rev, ds_id = _load_first(['OpenEvals/SimpleQA'], 'test')
    rows = _subset(ds, n)
    gens = _gen([r['problem'] for r in rows], 32, arms)
    res = {}
    for a in arms:
        hits = sum(1 for r, comp in zip(rows, gens[a['name']]) if _norm(comp.split('\n')[0]) == _norm(str(r['answer'])))
        res[a['name']] = {'acc': hits / max(1, len(rows)), 'n': len(rows)}
    return 'simpleqa', res, {'dataset': ds_id, 'rev': str(rev), 'n': len(rows)}
def t_mmlupro(arms, budget_ts, n=2000):
    ds, rev = _load_split('TIGER-Lab/MMLU-Pro', 'test')
    import random
    bycat = {}
    for r in ds:
        bycat.setdefault(r.get('category', '?'), []).append(r)
    rows = []
    per = max(1, n // max(1, len(bycat)))
    for cat, rs in sorted(bycat.items()):
        rows += _subset(rs, per, seed=EVAL_SEED + zlib.crc32(cat.encode()) % 999)  # crc32: hash() is salted
    rows = _subset(rows, n)
    letters = [chr(65 + i) for i in range(10)]
    res = {}
    for a in arms:
        hits = tot = 0
        for r in rows:
            if time.perf_counter() > budget_ts:
                break
            opts = r['options'][:10]
            prompt = r['question'] + '\n' + '\n'.join('%s. %s' % (L, o) for L, o in zip(letters, opts)) + '\nAnswer:'
            lps = _logprob_cont(prompt, [' ' + L for L in letters[:len(opts)]], a)
            if int(r['answer_index']) == max(range(len(opts)), key=lambda i: lps[i]):
                hits += 1
            tot += 1
        res[a['name']] = {'acc': hits / max(1, tot), 'n': tot}
    return 'mmlu-pro', res, {'dataset': 'TIGER-Lab/MMLU-Pro', 'rev': str(rev), 'n': len(rows), 'scoring': 'logprob-letter'}
def dump_perblock(arms, val_full, outpath):
    # paired per-block summed NLLs for bootstrap_nll.py
    import torch
    recs = []
    n = len(val_full)
    blocks = {a['name']: [] for a in arms}
    with torch.inference_mode():
        for bi, blk in enumerate(val_full):
            b = blk.unsqueeze(0)
            sums = {}
            for a in arms:
                a['inj'].set_memory(a['store'].lookup(addresses(b.cpu())).to('cuda') if a['store'] is not None else None)
                lg = model(input_ids=b.to('cuda'), use_cache=False).logits
                s = F.cross_entropy(lg[:, :-1].float().reshape(-1, lg.shape[-1]), b.to('cuda')[:, 1:].reshape(-1), reduction='sum').item()
                sums[a['name']] = s
            rec = {'block': bi, 'n': int(b[:, 1:].numel())}
            rec.update(sums)
            recs.append(rec)
    assert len(recs) == n and all(set(r) == {'block', 'n'} | {a['name'] for a in arms} for r in recs)
    Path(outpath).write_text('\n'.join(json.dumps(r) for r in recs))
    print('perblock wrote %d blocks -> %s' % (len(recs), outpath), flush=True)
    return recs
print('eval harness ready (6 tasks staged, perblock dumper)')

In [ ]:
# Stage A run: 4x500K controlled + exploratory REAL-1M. Eval only, no training gates in this notebook.
# Stage A controlled eval (NO training): 4x500K arms + exploratory REAL-1M
# Order: per-block NLL -> MMLU-Pro -> HumanEval+ -> SimpleQA (most expensive last, subset auto-reduced).
# Identical prompts/tokenizer/precision/decoding/seeds/batching/stopping for all arms.
import hashlib, json, re, subprocess, sys, time, zlib
import torch, torch.nn.functional as F
from pathlib import Path

T0 = time.perf_counter()
EOUT = Path('/kaggle/working/eval-stageA'); EOUT.mkdir(parents=True, exist_ok=True)
EFI_MOUNT = Path('/kaggle/input/qwen-ple-reader-checkpoints')
DLDIR = Path('/kaggle/working/ckpt-dl'); DLDIR.mkdir(parents=True, exist_ok=True)
need_pairs = [('reader-control-random-r1-500224.pt', 'reader-control-random-r1-500224.run.json', True),
              ('reader-control-permuted-r1-500224.pt', 'reader-control-permuted-r1-500224.run.json', True),
              ('real-500k-r1.reader.safetensors', 'real-500k-r1.run.json', False),
              ('real-1m-r1.reader.safetensors', 'real-1m-r1.run.json', False)]
def _mount_pair_ok(pt, runf, need_sha):
    try:
        run = json.loads((EFI_MOUNT / runf).read_text())
        ok = isinstance(run, dict) and 'token_count' in run and (EFI_MOUNT / pt).exists()
        if need_sha:
            ok = ok and 'sha256' in run
        if not ok:
            print('mount pair invalid: ' + pt, flush=True)
        return ok
    except Exception as e:
        print('mount pair error: ' + pt + ' ' + str(e)[:120], flush=True)
        return False
if EFI_MOUNT.exists() and all(_mount_pair_ok(p, r, s) for p, r, s in need_pairs):
    CKDIR = EFI_MOUNT
    print('checkpoints: mounted dataset (schema-validated)', flush=True)
else:
    _env = dict(os.environ); _env['KAGGLE_API_TOKEN'] = secret_value_1
    for _f in ['protocol.json',
               'reader-control-random-r1-500224.pt', 'reader-control-random-r1-500224.run.json',
               'reader-control-random-r1-500224.metrics.json',
               'reader-control-permuted-r1-500224.pt', 'reader-control-permuted-r1-500224.run.json',
               'reader-control-permuted-r1-500224.metrics.json',
               'real-500k-r1.reader.safetensors', 'real-500k-r1.run.json', 'real-500k-r1.metrics.json',
               'real-1m-r1.reader.safetensors', 'real-1m-r1.run.json', 'real-1m-r1.metrics.json']:
        _r = subprocess.run([sys.executable, '-m', 'kaggle', 'datasets', 'download', '-d',
                             'ninnix/qwen-ple-reader-checkpoints', '-f', _f, '-p', str(DLDIR), '--force'],
                            capture_output=True, text=True, env=_env, timeout=1200)
        assert _r.returncode == 0 and (DLDIR / _f).exists(), 'checkpoint download failed: ' + _f + ' rc=%d out=%s err=%s' % (_r.returncode, (_r.stdout or '')[-300:], (_r.stderr or '')[-300:])
    CKDIR = DLDIR
    print('checkpoints: api download (mount missing)', flush=True)
timings = {}
def _mark(name):
    timings[name] = round(time.perf_counter() - T0, 1)
    print('[t=%ds] %s' % (timings[name], name), flush=True)

_proto = json.loads((CKDIR / 'protocol.json').read_text())
def _file_sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
def _load_state(pt_name, run_name, expect_tokens):
    run = json.loads((CKDIR / run_name).read_text())
    h = _file_sha(CKDIR / pt_name)
    assert h == run['sha256'], 'bytes mismatch: ' + pt_name
    assert run.get('token_count') == expect_tokens, 'budget mismatch: ' + pt_name
    d = torch.load(str(CKDIR / pt_name), map_location='cpu')
    return d['reader'], h
def _load_sf(pt_name, run_name, expect_tokens):
    from safetensors.torch import load_file
    run = json.loads((CKDIR / run_name).read_text())
    assert run.get('token_count') == expect_tokens, 'budget mismatch: ' + run_name
    assert (CKDIR / pt_name).exists(), 'missing: ' + pt_name
    return load_file(str(CKDIR / pt_name), device='cpu'), 'e2e-checked-below'
_cksha = {}
_arms = []
def _mk(name, state, store):
    inj = ReaderInjection(model, [2, 8], C.MEM_DIM, C.HIDDEN, 1, C.GAMMA_INIT).to('cuda')
    if state is not None:
        inj.load_state_dict({k: v.to('cuda') for k, v in state.items()})
    _arms.append({'name': name, 'inj': inj, 'store': store})
_mk('disabled', None, None)
_st, _h = _load_state('reader-control-random-r1-500224.pt', 'reader-control-random-r1-500224.run.json', 500224)
_cksha['random'] = _h
_mk('random', _st, RandomPLE(_proto['calibrated_head_means'], _proto['calibrated_head_stds'], seed=0))
del _st
_mark('arms: disabled+random ready')

def _build_val_compact(pdir, perm_seed):
    need = True
    if (pdir / 'compact.json').exists():
        try:
            pm = json.loads((pdir / 'compact.json').read_text())
            need = not (pm.get('token_val') == C.VAL_FULL_TOKENS and pm.get('permutation_seed', None) == perm_seed
                        and pm.get('val_sha256') == _mF['tokens_sha256'])
        except Exception:
            need = True
    if need:
        from array import array as _array
        from safetensors import safe_open as _so
        _ms = MountPLE()
        _vv, _ = load_validation('full')
        _ad = ngram_indices(_vv)
        if perm_seed is not None:
            _ad = permute_addresses(_ad, seed=perm_seed)
        _uq = torch.unique(_ad.reshape(-1))
        pdir.mkdir(parents=True, exist_ok=True)
        _af = (pdir / 'addrs.u32').open('wb'); _rf = (pdir / 'rows.u8').open('wb')
        _pa = torch.div(_uq, C.ROWS_PER_PART, rounding_mode='floor'); _lo = _uq % C.ROWS_PER_PART
        _bf = {}
        for _p in torch.unique(_pa).tolist():
            _bf.setdefault(str(_ms.part_paths[_p]), []).append(_p)
        _n = 0; _prev = -1
        for _fp in sorted(_bf):
            with _so(_fp, framework='pt', device='cpu') as _fh:
                for _p in sorted(_bf[_fp]):
                    _full = _fh.get_slice(MountPLE.tensor_name(_p))[:]
                    _pos = torch.nonzero(_pa == _p).flatten()
                    _au = _uq.index_select(0, _pos)
                    assert int(_au[0]) > _prev; _prev = int(_au[-1])
                    _af.write(_array('I', _au.tolist()).tobytes())
                    _rf.write(bytes(_full.index_select(0, _lo.index_select(0, _pos)).view(torch.uint8).flatten().tolist()))
                    _n += _pos.numel()
        _af.close(); _rf.close()
        (pdir / 'compact.json').write_text(json.dumps({'format': 'qwen-ple-compact-val', 'version': 1,
            'token_val': int(_vv.numel()), 'address_count': _n, 'row_dim': C.ROW_DIM, 'scale': float(_ms.scale),
            'ple_revision': _ms.ple_revision, 'val_sha256': _mF['tokens_sha256'],
            'permutation_seed': perm_seed}, indent=2))
        _cc = CompactPLE(pdir)
        _g = torch.Generator().manual_seed(0)
        _samp = _uq[torch.randint(0, len(_uq), (2048,), generator=_g)].reshape(128, 16)
        _d = (_cc.lookup(_samp) - _ms.lookup(_samp)).abs().max().item()
        print('val-compact equivalence max|diff|:', _d); assert _d == 0.0
    return CompactPLE(pdir)
val_fast, _mf = load_validation('fast'); val_full, _mF = load_validation('full')
_cval = _build_val_compact(WORK / 'compact-val', None)
_cperm = _build_val_compact(WORK / 'compact-perm-val', 777)
_st, _h = _load_state('reader-control-permuted-r1-500224.pt', 'reader-control-permuted-r1-500224.run.json', 500224)
_cksha['permuted'] = _h
_mk('permuted', _st, PermutedPLE(_cperm, seed=777))
del _st
_st, _h = _load_sf('real-500k-r1.reader.safetensors', 'real-500k-r1.run.json', 500224)
_cksha['real'] = _h
_mk('real', _st, _cval)
del _st
_st, _h = _load_sf('real-1m-r1.reader.safetensors', 'real-1m-r1.run.json', 1000448)
_cksha['real-1m'] = _h
_mk('real-1m', _st, _cval)
del _st
_mark('arms: all 5 ready (4x500K controlled + REAL-1M exploratory)')

# Stage A gen stores: full-coverage (MountPLE-backed) for downstream prompts.
# NLL stores from part 1 cover only val addresses; prompt addresses need the mounted PLE.
_mstore = MountPLE()
_byname = {a['name']: a for a in _arms}
_byname['random']['gstore'] = _byname['random']['store']
_byname['permuted']['gstore'] = PermutedPLE(_mstore, seed=777)
_byname['real']['gstore'] = _mstore
_byname['real-1m']['gstore'] = _mstore
_byname['disabled']['gstore'] = None
# Spot-check: NLL store == gen store on 4 val blocks (exact, max|diff| 0.0), random deterministic.
_chk_blocks = [val_full[i].unsqueeze(0) for i in (0, 7, 511, 1023)]
for _a in _arms:
    if _a['name'] == 'disabled':
        continue
    _adds = addresses(_chk_blocks[0].cpu())
    assert torch.equal(_a['store'].lookup(_adds), _a['store'].lookup(_adds)), 'nondeterministic store: ' + _a['name']
for _b in _chk_blocks:
    _ad = addresses(_b.cpu())
    assert (_byname['real']['store'].lookup(_ad) - _mstore.lookup(_ad)).abs().max().item() == 0.0, 'real compact/mount drift'
    assert (_byname['permuted']['store'].lookup(_ad) - _byname['permuted']['gstore'].lookup(_ad)).abs().max().item() == 0.0, 'perm compact/mount drift'
print('gen-store equivalence ok (real + permuted, 4 val blocks)', flush=True)
_mark('gen stores ready')

# Inference primitives: identical batching/decoding for all arms. Greedy everywhere.
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = C.EOS
GEN = {}  # arm -> list of outputs per call (filled by runners for persistence)

def _solo(active, mem):
    # Exactly one injection may hold memory: hooks share layers, stale memories cross-talk and crash.
    for _o in _arms:
        _o['inj'].set_memory(mem if _o is active else None)

def _gen_mem(active, gs, ids, mask, raw_lens, max_new, pad):
    # Greedy decode with KV cache; per-step position-exact memory (prefill uses prompt memory,
    # each decode step uses the current last position's rows). Matches the hook shape contract.
    import torch
    B, Lp = ids.shape
    device = 'cuda'
    ids_c = ids.to(device)
    attn = mask.to(device)
    _solo(active, gs.lookup(addresses(ids)).to(device))
    out = model(input_ids=ids_c, attention_mask=attn, use_cache=True)
    past = out.past_key_values
    tails = [ids[b, Lp - raw_lens[b]:].tolist() for b in range(B)]
    gen_toks = [[] for _ in range(B)]
    done = [False] * B
    nxt = out.logits[:, -1].argmax(-1).tolist()
    for b in range(B):
        if nxt[b] == pad:
            done[b] = True
        else:
            gen_toks[b].append(nxt[b]); tails[b].append(nxt[b])
    cur = torch.tensor(nxt, dtype=torch.long, device=device).unsqueeze(1)
    step = 1
    while step < max_new and not all(done):
        ms = []
        for b in range(B):
            w = tails[b][-64:]
            assert len(w) >= 3, 'prompt too short for trigram addressing'
            ms.append(gs.lookup(ngram_indices(torch.tensor([w]))[:, -1:, :]))
        _solo(active, torch.cat(ms, dim=0).to(device))
        attn = torch.cat([attn, torch.ones(B, 1, dtype=torch.long, device=device)], dim=1)
        out = model(input_ids=cur, attention_mask=attn, past_key_values=past, use_cache=True)
        past = out.past_key_values
        nxt = out.logits[:, -1].argmax(-1).tolist()
        cur = torch.tensor(nxt, dtype=torch.long, device=device).unsqueeze(1)
        for b in range(B):
            if not done[b]:
                if nxt[b] == pad:
                    done[b] = True
                else:
                    gen_toks[b].append(nxt[b]); tails[b].append(nxt[b])
        step += 1
    return [tokenizer.decode(ts, skip_special_tokens=True) for ts in gen_toks]

def _gen(prompts, max_new):
    # Greedy completions per arm. Disabled uses the generate fast path (all hooks passthrough);
    # memory arms use the stepwise loop above (generate() reuses stale hook memory across steps).
    import torch
    _prep_tok()
    pad = tokenizer.pad_token_id
    out = {a['name']: [] for a in _arms}
    for i in range(0, len(prompts), C.EVAL_BS):
        chunk = prompts[i:i + C.EVAL_BS]
        enc = tokenizer(chunk, return_tensors='pt', padding=True, add_special_tokens=False)
        ids, mask = enc['input_ids'], enc['attention_mask']
        raw_lens = mask.sum(dim=1).tolist()
        for a in _arms:
            gs = a['gstore']
            with torch.inference_mode():
                if gs is None:
                    _solo(a, None)
                    g = model.generate(input_ids=ids.to('cuda'), attention_mask=mask.to('cuda'),
                                       max_new_tokens=max_new, do_sample=False,
                                       pad_token_id=pad, use_cache=True)
                    outs = tokenizer.batch_decode(g[:, ids.shape[1]:], skip_special_tokens=True)
                else:
                    outs = _gen_mem(a, gs, ids, mask, raw_lens, max_new, pad)
            out[a['name']].extend(outs)
    return out

def _logprob_cont(prompt, conts, arm):
    p = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)['input_ids']
    outs = []
    with torch.inference_mode():
        for c in conts:
            t = tokenizer(c, return_tensors='pt', add_special_tokens=False)['input_ids']
            ids = torch.cat([p, t], dim=1)
            gs = arm['gstore']
            _solo(arm, gs.lookup(addresses(ids)).to('cuda') if gs is not None else None)
            lg = model(input_ids=ids.to('cuda'), use_cache=False).logits.float()
            lp = torch.log_softmax(lg[0, p.shape[1] - 1:-1], dim=-1)
            outs.append(lp.gather(1, t[0].to('cuda').unsqueeze(1)).sum().item())
    return outs

def _exec_check(code, timeout_s=10):
    try:
        r = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, timeout=timeout_s)
        return r.returncode == 0, (r.stderr or '')[-200:]
    except subprocess.TimeoutExpired:
        return False, 'timeout'

def _norm(s):
    s = s.lower().strip()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return re.sub(r'[^a-z0-9]', '', s)

def _ds_rev(ds_id):
    from huggingface_hub import HfApi
    try:
        return HfApi(token=secret_value_0).dataset_info(ds_id).sha
    except Exception:
        return None

def _load_split(ds_id, split):
    from datasets import load_dataset
    rev = _ds_rev(ds_id)
    kw = {'revision': rev} if rev else {}
    return load_dataset(ds_id, split=split, **kw), rev

def _subset(rows, n, seed):
    import random
    rows = list(rows)
    if len(rows) <= n:
        return rows
    rng = random.Random(seed)
    idx = list(range(len(rows)))
    rng.shuffle(idx)
    return [rows[i] for i in sorted(idx[:n])]

# Eval determinism: identical greedy outputs on repeat (disabled arm, 20 prompts).
_det_p = ['The capital of France is', '2+2=', 'def f(x):', 'The quick brown fox'] * 5
_d1 = _gen(_det_p, 8)['disabled']
_d2 = _gen(_det_p, 8)['disabled']
assert _d1 == _d2, 'eval path not deterministic'
print('eval determinism ok', flush=True)
_mark('inference primitives ready')

# Benchmarks: MMLU-Pro (logprob-letter) -> HumanEval+ (batched gen + exec) -> SimpleQA (batched gen).
# Each returns (name, {arm: {metrics}}, meta, predictions[{arm,idx,prompt,output,score}]).
PRED = []

def t_mmlupro(n):
    ds, rev = _load_split('TIGER-Lab/MMLU-Pro', 'test')
    bycat = {}
    for r in ds:
        bycat.setdefault(r.get('category', '?'), []).append(r)
    rows = []
    per = max(1, n // max(1, len(bycat)))
    for cat, rs in sorted(bycat.items()):
        rows += _subset(rs, per, C.EVAL_SEED + zlib.crc32(cat.encode()) % 999)
    rows = _subset(rows, n, C.EVAL_SEED)
    letters = [chr(65 + i) for i in range(10)]
    res = {}
    for a in _arms:
        hits = tot = 0
        for j, r in enumerate(rows):
            opts = r['options'][:10]
            prompt = r['question'] + '\n' + '\n'.join('%s. %s' % (L, o) for L, o in zip(letters, opts)) + '\nAnswer:'
            lps = _logprob_cont(prompt, [' ' + L for L in letters[:len(opts)]], a)
            pred = max(range(len(opts)), key=lambda i: lps[i])
            ok = int(pred == int(r['answer_index']))
            hits += ok; tot += 1
            PRED.append({'task': 'mmlu-pro', 'arm': a['name'], 'idx': j, 'prompt': prompt,
                         'output': letters[pred], 'score': ok})
        res[a['name']] = {'acc': hits / max(1, tot), 'n': tot}
    return 'mmlu-pro', res, {'dataset': 'TIGER-Lab/MMLU-Pro', 'rev': str(rev), 'n': len(rows), 'scoring': 'logprob-letter'}

def t_humaneval():
    try:
        from evalplus.data import get_human_eval_plus
        probs = get_human_eval_plus()
        ds_id, ds_rev, plus = 'evalplus/HumanEvalPlus', 'builtin', True
    except Exception:
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'evalplus'], timeout=600, check=False)
            from evalplus.data import get_human_eval_plus
            probs = get_human_eval_plus()
            ds_id, ds_rev, plus = 'evalplus/HumanEvalPlus', 'builtin', True
        except Exception:
            ds, rev = _load_split('openai_humaneval', 'test')
            probs = {r['task_id']: {'prompt': r['prompt'], 'test': r['test'], 'entry_point': r['entry_point']} for r in ds}
            ds_id, ds_rev, plus = 'openai_humaneval', rev, False
    items = sorted(probs.items())
    gens = _gen([p['prompt'] for _, p in items], 512)
    res = {}
    for a in _arms:
        ok = te = 0
        for j, ((tid, p), comp) in enumerate(zip(items, gens[a['name']])):
            code = p['prompt'] + '\n' + comp + '\n' + p['test'] + '\ncheck(' + p['entry_point'] + ')'
            passed = False
            try:
                passed, err = _exec_check(code)
                ok += int(passed)
            except Exception:
                te += 1
            PRED.append({'task': 'humaneval+', 'arm': a['name'], 'idx': j, 'prompt': p['prompt'],
                         'output': comp, 'score': int(passed)})
        res[a['name']] = {'pass': ok, 'n': len(items), 'acc': ok / max(1, len(items)), 'exec_errors': te}
    return 'humaneval+', res, {'dataset': ds_id, 'rev': str(ds_rev), 'plus': plus, 'n': len(items)}

def _load_first(cands, split):
    errs = []
    for _cid in cands:
        try:
            _ds, _rev = _load_split(_cid, split)
            print('dataset resolved: ' + _cid, flush=True)
            return _ds, _rev, _cid
        except Exception as e:
            errs.append(_cid + ': ' + str(e)[:120])
    raise RuntimeError('no candidate resolved: ' + ' | '.join(errs))
def t_simpleqa(n):
    ds, rev, ds_id = _load_first(['OpenEvals/SimpleQA'], 'test')
    rows = _subset(ds, n, C.EVAL_SEED)
    gens = _gen([r['problem'] for r in rows], 32)
    res = {}
    for a in _arms:
        hits = 0
        for j, (r, comp) in enumerate(zip(rows, gens[a['name']])):
            ok = int(_norm(comp.split('\n')[0]) == _norm(str(r['answer'])))
            hits += ok
            PRED.append({'task': 'simpleqa', 'arm': a['name'], 'idx': j, 'prompt': r['problem'],
                         'output': comp, 'score': ok})
        res[a['name']] = {'acc': hits / max(1, len(rows)), 'n': len(rows)}
    return 'simpleqa', res, {'dataset': ds_id, 'rev': str(rev), 'n': len(rows)}
print('benchmarks defined', flush=True)

# Per-block NLLs (frozen full val, identical blocks) -> projection/adaptive subsets -> benchmarks -> bootstrap -> persist.
results = {'arms': [a['name'] for a in _arms], 'benchmarks': {}}

# 1. Per-block NLLs.
recs = []
with torch.inference_mode():
    for bi, blk in enumerate(val_full):
        b = blk.unsqueeze(0)
        rec = {'block': bi, 'n': int(b[:, 1:].numel())}
        for a in _arms:
            _solo(a, a['store'].lookup(addresses(b.cpu())).to('cuda') if a['store'] is not None else None)
            lg = model(input_ids=b.to('cuda'), use_cache=False).logits
            rec[a['name']] = F.cross_entropy(lg[:, :-1].float().reshape(-1, lg.shape[-1]),
                                             b.to('cuda')[:, 1:].reshape(-1), reduction='sum').item()
        recs.append(rec)
(EOUT / 'perblock-nll.jsonl').write_text('\n'.join(json.dumps(r) for r in recs))
_mark('perblock NLLs: %d blocks x %d arms' % (len(recs), len(_arms)))
_exp = {'disabled': 2.90558,
        'random': json.loads((CKDIR / 'reader-control-random-r1-500224.metrics.json').read_text())['val_loss'],
        'permuted': json.loads((CKDIR / 'reader-control-permuted-r1-500224.metrics.json').read_text())['val_loss'],
        'real': json.loads((CKDIR / 'real-500k-r1.metrics.json').read_text())['val_loss'],
        'real-1m': json.loads((CKDIR / 'real-1m-r1.metrics.json').read_text())['val_loss']}
for _a, _e in _exp.items():
    _got = sum(r[_a] / r['n'] for r in recs) / len(recs)
    print('arm %s perblock-mean %.5f expected %.5f' % (_a, _got, _e), flush=True)
    assert abs(_got - _e) < 0.002, 'arm/metric mismatch (wrong weights or store): ' + _a

# 2. Projection: MMLU-Pro is cheapest informative probe of benchmark rate; SimpleQA decides subsets.
_t = time.perf_counter()
_n_pred = len(PRED)
_mn, _mres, _mmeta = t_mmlupro(50)
del PRED[_n_pred:]  # probe predictions excluded from persisted outputs
_mmlu_rate = (time.perf_counter() - _t) / 50.0
_t = time.perf_counter()
_sq_rows, _sqrev, _sq_id = _load_first(['OpenEvals/SimpleQA'], 'test')
_sq_probe = _subset(_sq_rows, 10, C.EVAL_SEED)
_g = _gen([r['problem'] for r in _sq_probe], 32)
_sqa_rate = (time.perf_counter() - _t) / 10.0
_h_est, _b_est = 1500.0, (time.perf_counter() - T0)
proj = _b_est + 5 * (C.EVAL_MMLU_N * _mmlu_rate + C.EVAL_SQA_N * _sqa_rate) + _h_est
print('rates: mmlu %.2fs/item humaneval-est %.0fs simpleqa %.2fs/item | projected total %.0fs (target %d)' % (
    _mmlu_rate, _h_est, _sqa_rate, proj, C.EVAL_TARGET_S), flush=True)
MN, SN = C.EVAL_MMLU_N, C.EVAL_SQA_N
while proj > C.EVAL_TARGET_S and (SN > 250 or MN > 250):
    if SN > 250:
        SN //= 2
    elif MN > 250:
        MN //= 2
    proj = _b_est + 5 * (MN * _mmlu_rate + SN * _sqa_rate) + _h_est
print('subsets: mmlu-pro %d simpleqa %d (projected %.0fs)' % (MN, SN, proj), flush=True)

# 3. Benchmarks (HumanEval+ complete, never subset).
for _fn, _nn in ((lambda: t_mmlupro(MN), None), (t_humaneval, None), (lambda: t_simpleqa(SN), None)):
    if (time.perf_counter() - T0) > C.EVAL_HARD_S - 1800:
        print('HARD-GUARD: stopping before next benchmark; persisting partial', flush=True)
        break
    _t = time.perf_counter()
    try:
        _tname, _res, _meta = _fn()
        results['benchmarks'][_tname] = {'status': 'ok', 'scores': _res, 'meta': _meta,
                                         'seconds': round(time.perf_counter() - _t, 1)}
    except Exception as e:
        results['benchmarks'][getattr(_fn, '__name__', '?')] = {'status': 'error', 'error': str(e)[:300]}
    (EOUT / 'benchmarks-partial.json').write_text(json.dumps(results['benchmarks'], indent=2))
    _mark('benchmark done')
(EOUT / 'predictions.jsonl').write_text('\n'.join(json.dumps(p) for p in PRED))

# 4. Paired bootstrap 95% CIs (same algorithm as tools/bootstrap_nll.py, seed pinned).
import random as _r
def _boot(a, b, n_boot=10000, seed=1234):
    dd = [(r[a] - r[b]) / r['n'] for r in recs]
    rng = _r.Random(seed)
    n = len(dd)
    reps = sorted(sum(dd[rng.randrange(n)] for _ in range(n)) / n for _ in range(n_boot))
    ge = sum(1 for x in reps if x >= 0.0)
    le = sum(1 for x in reps if x <= 0.0)
    return {'mean': sum(dd) / n, 'lo': reps[int(0.025 * n_boot)], 'hi': reps[int(0.975 * n_boot) - 1],
            'p': min(1.0, 2.0 * min(ge, le) / n_boot)}
CONTRASTS = [('real', 'disabled'), ('real', 'random'), ('real', 'permuted'), ('real-1m', 'real')]
boot = {}
for a, b in CONTRASTS:
    boot['%s-vs-%s' % (a, b)] = _boot(a, b)
    m = boot['%s-vs-%s' % (a, b)]
    print('%s-vs-%s: mean %+.5f 95%%CI [%+.5f, %+.5f] p=%.4g %s' % (
        a.upper(), b.upper(), m['mean'], m['lo'], m['hi'], m['p'], 'ROBUST' if m['hi'] < 0 else 'NOT-ROBUST'), flush=True)
(EOUT / 'bootstrap.json').write_text(json.dumps(boot, indent=2))

# 5. Persist everything; close injections.
(EOUT / 'config.json').write_text(json.dumps({
    'seeds': {'eval': C.EVAL_SEED, 'ple': C.SEED}, 'batch': C.EVAL_BS, 'subsets': {'mmlu-pro': MN, 'simpleqa': SN},
    'decoding': 'greedy do_sample=False', 'max_new': {'simpleqa': 32, 'humaneval+': 512},
    'precision': 'backbone-fp16 reader-FP32 ple-FP32', 'arms': results['arms'],
    'prompts': 'mmlu-pro letter template / simpleqa problem / humaneval+ prompt; base model, no chat template'}, indent=2))
(EOUT / 'checkpoint-shas.json').write_text(json.dumps(_cksha, indent=2))
(EOUT / 'validation-sha.json').write_text(json.dumps({'full': _mF['tokens_sha256'], 'fast': _mf['tokens_sha256']}, indent=2))
(EOUT / 'timings.json').write_text(json.dumps(timings, indent=2))
(EOUT / 'summary.json').write_text(json.dumps(results, indent=2))
for a in _arms:
    a['inj'].close()
_mark('STAGE A COMPLETE: persisted predictions, per-item scores, per-block NLLs, configs, SHAs, timings, bootstrap')
print('NO training ran in this job (eval-only kernel). heavy reasoning/code suites deferred to a later job.', flush=True)
